# 📑 Smart Document Reader with Dual-Provider (Gemini + Groq)

Turn any document into a smart assistant — ask for summaries, key facts, or detailed explanations.

### Dual-Engine Resilience:
This pipeline uses **both Google Gemini and Groq** with automatic failover — if one provider hits a rate limit (e.g. 429), it automatically answers using the other!

## 1. Install Dependencies

In [ ]:
%pip install -q --upgrade \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-google-genai \
    langchain-groq \
    faiss-cpu \
    pypdf \
    tiktoken \
    gradio

## 2. Set API Keys (Both Gemini & Groq)

In [ ]:
import os

# Set Groq and Google Gemini API keys
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")
os.environ["GOOGLE_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

print("✅ Both Gemini and Groq keys configured.")

## 3. Upload or Select PDF

In [ ]:
try:
    from google.colab import files
    print("Click below to choose your PDF file:")
    uploaded = files.upload()
    pdf_filename = list(uploaded.keys())[0]
except ImportError:
    # Fallback for local files
    pdf_filename = "sample.pdf"  # Put your PDF path here

print(f"✅ Selected Document: {pdf_filename}")

## 4. Read and Prepare the Document

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_filename)
pages = loader.load()
print(f"📄 Total pages loaded: {len(pages)}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(pages)
print(f"✂️ Ready: {len(chunks)} sections prepared.")

## 5. Index Document with Gemini Embeddings & FAISS

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

print("⏳ Indexing document... please wait a few seconds")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

vector_store = FAISS.from_documents(chunks, embeddings)
print(f"✅ Index ready! Total sections: {vector_store.index.ntotal}")

## 6. Setup Question-Answering with Failover

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Initialize both LLMs
gemini_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", google_api_key=os.environ["GEMINI_API_KEY"])
groq_llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=os.environ["GROQ_API_KEY"])

prompt = PromptTemplate(
    template="""You are a helpful, clear, and friendly document assistant. Your job is to answer questions based on the document provided in a way that is simple, direct, and easy for anyone to understand.

Rules:
1. Explain in simple, plain everyday language. Avoid confusing technical jargon.
2. If asked for a summary or key points, organize your answer with clear bullet points.
3. Answer strictly using facts from the Document Context below.
4. If the document does not contain the answer, say honestly: "This document does not contain information about that."

Conversation History:
{chat_history}

Document Context:
{context}

User Question:
{question}

Answer:""",
    input_variables=["chat_history", "context", "question"]
)

gemini_chain = prompt | gemini_llm | StrOutputParser()
groq_chain = prompt | groq_llm | StrOutputParser()
chat_history_store = []

def ask(question):
    history_text = ""
    for human, ai in chat_history_store:
        history_text += f"Human: {human}\nAssistant: {ai}\n"

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    payload = {"chat_history": history_text, "context": context, "question": question}

    # Dual-provider failover: try Gemini, fallback to Groq if rate-limited
    try:
        answer = gemini_chain.invoke(payload)
    except Exception as e:
        print(f"⚠️ Gemini rate-limited or busy, failing over to Groq... ({e})")
        answer = groq_chain.invoke(payload)

    chat_history_store.append((question, answer))
    return answer, docs

print("✅ Dual-provider RAG assistant ready!")

## 7. Test Query

In [ ]:
test_query = "What is the main topic of this document?"
answer, sources = ask(test_query)

print(f"💬 Answer:\n{answer}\n")
print("📍 Found in your document:")
for doc in sources:
    page_raw = doc.metadata.get('page', 0)
    page_num = int(page_raw) + 1 if isinstance(page_raw, (int, float)) else str(page_raw)
    snippet = doc.page_content[:150].replace('\n', ' ').strip()
    print(f"• Page {page_num}: \"{snippet}...\"")

## 8. Launch Interactive Chat Web Interface

In [ ]:
import gradio as gr

def chat_handler(question, history):
    history_list = list(history or [])
    if not question or not question.strip():
        return history_list, ""

    answer, docs = ask(question)

    sources = []
    for doc in docs:
        page_raw = doc.metadata.get("page", 0)
        page_num = int(page_raw) + 1 if isinstance(page_raw, (int, float)) else str(page_raw)
        snippet = doc.page_content[:150].replace("\n", " ").strip()
        sources.append(f"• **Page {page_num}**: _{snippet}..._")

    sources_text = "\n".join(sources)
    full_answer = f"{answer}\n\n---\n📍 **Found in your document:**\n{sources_text}"
    history_list.append({"role": "user", "content": question})
    history_list.append({"role": "assistant", "content": full_answer})
    return history_list, ""

with gr.Blocks(title="Smart Document Reader") as demo:
    gr.Markdown("# 📑 Smart Document Reader\nTurn any document into a smart assistant — ask for summaries, key facts, or detailed explanations.")

    chatbot = gr.Chatbot(
        label="Chat",
        height=450,
        placeholder="👋 Ask me any question about your document!"
    )

    with gr.Row():
        question_box = gr.Textbox(
            placeholder="Type your question here and press Enter...",
            label="Your Question",
            scale=8,
            autofocus=True
        )
        submit_btn = gr.Button("Ask ➔", variant="primary", scale=1)

    gr.ClearButton(components=[chatbot, question_box], value="🗑️ Clear Chat")

    question_box.submit(fn=chat_handler, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    submit_btn.click(fn=chat_handler, inputs=[question_box, chatbot], outputs=[chatbot, question_box])

demo.launch(theme=gr.themes.Soft(), debug=True)